In [1]:
from sklearn.mixture import GaussianMixture
import joblib

In [2]:
import torch

def build_features(per_ch_err, latent):
    """
    per_ch_err: [B, C]
    latent:     [B, 32]

    Returns:
        features: [B, C + 32]
    """
    return torch.cat([per_ch_err, latent], dim=1)

In [3]:
from torch.utils.data import Dataset

class SubsystemDataset(Dataset):
    def __init__(self, per_ch_err, latent, labels):
        """
        per_ch_err : Tensor [N, C]
        latent     : Tensor [N, 32]
        labels     : Tensor [N]
        """
        self.features = torch.cat([per_ch_err, latent], dim=1)
        self.labels = labels.long()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [4]:
import numpy as np
from sklearn.mixture import GaussianMixture

class GMMSubsystemClassifier:
    def __init__(self, n_classes=6, n_components=4):
        self.n_classes = n_classes
        self.models = [
            GaussianMixture(
                n_components=n_components,
                covariance_type="full",
                random_state=42
            )
            for _ in range(n_classes)
        ]

    def fit(self, X, y):
        """
        X: numpy array [N, D]
        y: numpy array [N]
        """
        for cls in range(self.n_classes):
            X_cls = X[y == cls]
            self.models[cls].fit(X_cls)

    def predict_logits(self, X):
        """
        Returns:
            logits: [N, 6]
        """
        scores = []
        for model in self.models:
            # log p(x | class)
            scores.append(model.score_samples(X))

        logits = np.stack(scores, axis=1)
        return logits

    def predict(self, X):
        logits = self.predict_logits(X)
        return np.argmax(logits, axis=1)

### Generate random Training Data 

In [5]:
import torch

N_train = 10_000
N_test = 2_000
C = 55
LATENT_DIM = 32
NUM_CLASSES = 6

# Generate labels first
labels_train = torch.randint(0, NUM_CLASSES, (N_train,))
labels_test = torch.randint(0, NUM_CLASSES, (N_test,))

# Create one prototype latent vector per subsystem
latent_centers = torch.randn(NUM_CLASSES, LATENT_DIM) * 3.0

# Each sample = subsystem center + Gaussian noise
latent_train = (
    latent_centers[labels_train]
    + 0.5 * torch.randn(N_train, LATENT_DIM)
)

latent_test = (
    latent_centers[labels_test]
    + 0.5 * torch.randn(N_test, LATENT_DIM)
)

# Similarly create subsystem-specific reconstruction errors
err_centers = torch.rand(NUM_CLASSES, C)

per_ch_err_train = (
    err_centers[labels_train]
    + 0.1 * torch.randn(N_train, C)
).clamp(min=0.0)

per_ch_err_test = (
    err_centers[labels_test]
    + 0.1 * torch.randn(N_test, C)
).clamp(min=0.0)

### Training end to end pipeline with Training Data (GMM & XGboost)

In [6]:
from xgboost import XGBClassifier

# ----------------------------------------
# Build features
# ----------------------------------------
X_train = torch.cat(
    [per_ch_err_train, latent_train],
    dim=1
).cpu().numpy()

X_test = torch.cat(
    [per_ch_err_test, latent_test],
    dim=1
).cpu().numpy()

y_train = labels_train.cpu().numpy()
y_test = labels_test.cpu().numpy()

# ----------------------------------------
# Train GMM
# ----------------------------------------
gmm = GMMSubsystemClassifier(
    n_classes=6,
    n_components=4
)

gmm.fit(X_train, y_train)

gmm_logits = gmm.predict_logits(X_test)
gmm_preds = np.argmax(gmm_logits, axis=1)

# ----------------------------------------
# Train XGBoost (Alpha)
# ----------------------------------------
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=6,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    eval_metric="mlogloss",
)

xgb_model.fit(X_train, y_train)

xgb_probs = xgb_model.predict_proba(X_test)
xgb_logits = np.log(xgb_probs + 1e-8)
xgb_preds = np.argmax(xgb_probs, axis=1)

### Export/Save GMM model 

In [7]:
import joblib

# Save the trained GMM classifier
joblib.dump(gmm, "gmm_subsystem_classifier.pkl")

print("GMM model saved.")

GMM model saved.


### Export/Save XGBoost model

In [8]:
import joblib

joblib.dump(xgb_model, "xgboost_subsystem_classifier.pkl")

print("XGBoost model saved.")

XGBoost model saved.


### MLP

In [9]:
import torch
import torch.nn as nn

class SubsystemMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_classes=6):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [12]:
# Features as tensors (for PyTorch)
X_train_tensor = torch.cat(
    [per_ch_err_train, latent_train],
    dim=1
)

X_test_tensor = torch.cat(
    [per_ch_err_test, latent_test],
    dim=1
)

y_train_tensor = labels_train.long()
y_test_tensor = labels_test.long()

In [13]:


model = SubsystemMLP(
    input_dim=X_train_tensor.shape[1],
    num_classes=6
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

for epoch in range(50):
    optimizer.zero_grad()

    logits = model(X_train_tensor)
    loss = criterion(logits, y_train_tensor)

    loss.backward()
    optimizer.step()

### LSTM

##### LSTM reuires a sequential number (think of like sliding window in relation a temporal-timing factor)

In [14]:
import torch

def make_sequences(X, y, seq_len=10):
    """
    X: [N, D]
    y: [N]

    returns:
        X_seq: [N-T+1, T, D]
        y_seq: [N-T+1]
    """
    X_seq = []
    y_seq = []

    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i+seq_len-1])  # label at last timestep

    return torch.stack(X_seq), torch.tensor(y_seq)

In [16]:
seq_len = 10

X_train_seq, y_train_seq = make_sequences(X_train_tensor, y_train_tensor, seq_len)
X_test_seq, y_test_seq = make_sequences(X_test_tensor, y_test_tensor, seq_len)

print(X_train_seq.shape)  # [N_seq, 10, 87]

torch.Size([9990, 10, 87])


### Define LSTM model

In [17]:
import torch
import torch.nn as nn

class SubsystemLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, num_classes=6):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x: [B, T, D]
        out, (h_n, c_n) = self.lstm(x)

        # last layer hidden state
        last_hidden = h_n[-1]  # [B, H]

        logits = self.fc(last_hidden)
        return logits

##### Train LSTM

In [18]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SubsystemLSTM(input_dim=X_train.shape[1]).to(device)

X_train_seq = X_train_seq.to(device)
y_train_seq = y_train_seq.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 20

for epoch in range(epochs):
    model.train()

    optimizer.zero_grad()

    logits = model(X_train_seq)
    loss = criterion(logits, y_train_seq)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

Epoch 1/20 | Loss: 1.7837
Epoch 2/20 | Loss: 1.7696
Epoch 3/20 | Loss: 1.7556
Epoch 4/20 | Loss: 1.7421
Epoch 5/20 | Loss: 1.7279
Epoch 6/20 | Loss: 1.7133
Epoch 7/20 | Loss: 1.6980
Epoch 8/20 | Loss: 1.6822
Epoch 9/20 | Loss: 1.6653
Epoch 10/20 | Loss: 1.6477
Epoch 11/20 | Loss: 1.6286
Epoch 12/20 | Loss: 1.6086
Epoch 13/20 | Loss: 1.5870
Epoch 14/20 | Loss: 1.5639
Epoch 15/20 | Loss: 1.5398
Epoch 16/20 | Loss: 1.5135
Epoch 17/20 | Loss: 1.4849
Epoch 18/20 | Loss: 1.4554
Epoch 19/20 | Loss: 1.4251
Epoch 20/20 | Loss: 1.3919


#### Model Evaluation

In [19]:
model.eval()
with torch.no_grad():
    logits = model(X_test_seq.to(device))
    preds = torch.argmax(logits, dim=1)

acc = (preds.cpu() == y_test_seq).float().mean()
print("LSTM accuracy:", acc.item())

LSTM accuracy: 0.9361808896064758


### PCMCI Model (Causal Discovery) (Not classifier)

###### PCMCI expects a T × D matrix

In [22]:
X_ts = X_train

In [23]:
from tigramite import data_processing as pp

dataframe = pp.DataFrame(X_ts)

### Define PCMCI model

In [24]:
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr

pcmci = PCMCI(
    dataframe=dataframe,
    cond_ind_test=ParCorr()
)

### Run causal Discovery

In [25]:
results = pcmci.run_pcmci(
    tau_max=5,        # max time lag
    pc_alpha=0.05     # significance threshold
)

### Extract Causal Graph

In [26]:
q_matrix = results["p_matrix"]
val_matrix = results["val_matrix"]

### Print strongest causal links

In [27]:
import numpy as np

D = X_train.shape[1]

links = []

for i in range(D):
    for j in range(D):
        for tau in range(1, 6):
            val = val_matrix[i, j, tau]
            p = q_matrix[i, j, tau]

            if p < 0.01:
                links.append((i, j, tau, val))

# sort strongest
links = sorted(links, key=lambda x: abs(x[3]), reverse=True)

print("Top causal relationships:")
for l in links[:10]:
    print(f"X{l[1]}(t-{l[2]}) → X{l[0]} | strength={l[3]:.4f}")

Top causal relationships:
X0(t-1) → X34 | strength=0.0429
X78(t-1) → X34 | strength=-0.0404
X80(t-2) → X55 | strength=0.0395
X35(t-1) → X34 | strength=-0.0395
X66(t-2) → X55 | strength=-0.0364
X50(t-1) → X47 | strength=-0.0361
X66(t-2) → X2 | strength=0.0356
X55(t-1) → X34 | strength=0.0351
X62(t-1) → X34 | strength=0.0347
X55(t-4) → X70 | strength=0.0340
